<a href="https://colab.research.google.com/github/harshav6/deepLearning/blob/main/AlexNet/alexnet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import wandb
import os

In [ ]:
wandb.init(project='alexnet-flowers-v2', config={
    "epochs": 10,
    "batch_size": 16,
    "learning_rate": 0.001,
    "architecture": "alexnet",
    "pretrained": True,
    "input_size": 128
})

config = wandb.config

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 1


wandb: You chose 'Create a W&B account'
wandb: Create an account here: https://wandb.ai/authorize?signup=true&ref=models
wandb: After creating your account, create a new API key and store it securely.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: harshavardhanv16 (harshavardhanv16-anil) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
transform = transforms.Compose([
    transforms.Resize((config.input_size, config.input_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

])

train_dir = "/content/flowers/flowers/train"
val_dir = "/content/flowers/flowers/val"

In [ ]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/flowers-20260820T043258Z-1-001.zip"
extract_path = "/content/flowers"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print(os.listdir(extract_path))

['flowers']


In [ ]:
!find /content/flowers -maxdepth 3 -type d | head -50

/content/flowers
/content/flowers/flowers
/content/flowers/flowers/train
/content/flowers/flowers/train/Sunflower
/content/flowers/flowers/train/Orchid
/content/flowers/flowers/train/Tulip
/content/flowers/flowers/train/Lilly
/content/flowers/flowers/train/Lotus
/content/flowers/flowers/val
/content/flowers/flowers/val/Sunflower
/content/flowers/flowers/val/Orchid
/content/flowers/flowers/val/Tulip
/content/flowers/flowers/val/Lilly
/content/flowers/flowers/val/Lotus


In [ ]:
train_dataset = datasets.ImageFolder(root=train_dir, transform=transform)
val_dataset = datasets.ImageFolder(root=val_dir, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config.batch_size)

In [ ]:
import os

print(os.listdir("/content/drive/MyDrive"))

['Colab Notebooks', 'flowchart.drawio', 'Presentation (1).pptx', 'Yearbook.gslides', 'Presentation.pptx', 'Untitled presentation.gslides', 'ES pre.gslides', 'Screenshot 2024-05-28 103939.png', 'Screenshot 2024-05-28 104100.png', 'Untitled document.gdoc', 'resume.gdoc', 'myResume.pdf', 'pic', 'basic.drawio', 'project.drawio', 'intern.drawio', 'resumeharsha (1).pdf', 'Resume__5_.pdf', 'MlResume (1).pdf', 'MlResume.pdf', 'Harsha_finalyear_thesis.pdf', 'Phase-2 Capstone Project Report FINAL .gdoc', 'harsha_bachelors_modulehandbook.pdf', '22.pdf', 'flowers-20260820T043258Z-1-001.zip']


In [ ]:
alexnet = models.alexnet(pretrained=config.pretrained)

# Freeze All Layers
for param in alexnet.parameters():
  param.requires_grad = False

# Replace final classifier Layer
alexnet.classifier[6] = nn.Linear(4096, 5)

# Only the new final Layer should be trainable
for param in alexnet.classifier[6].parameters():
  param.requires_grad = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
alexnet = alexnet.to(device)

# Watch the model's weights and gradients
wandb.watch(alexnet, log="all", log_freq=10)


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 129MB/s]


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(alexnet.parameters(), lr=config.learning_rate)

In [ ]:
def train_model(model, criterion, optimizer, train_loader, val_loader, epochs = 10):
  for epoch in range(epochs):
    model.train()
    train_correct = 0
    train_total = 0
    running_loss = 0.0

    print(f"\nEpoch {epoch+1}/{epochs}")
    print("-" * 30)

    for i, (images, labels) in enumerate(train_loader):
      images,labels = images.to(device), labels.to(device)

      optimizer.zero_grad()
      outputs = model(images)
      loss = criterion(outputs, labels)
      loss.backward()
      optimizer.step()

      running_loss += loss.item()
      _, preds = torch.max(outputs, 1)
      batch_correct = (preds == labels).sum().item()
      train_correct += batch_correct
      train_total += labels.size(0)

      if (i + 1) % 10 == 0:
                batch_acc = batch_correct / labels.size(0)
                print(f"[Batch {i+1}/{len(train_loader)}] Loss: {loss.item():.4f}, Batch Acc: {batch_acc:.4f}")

    train_acc = train_correct / train_total
    wandb.log({"epoch": epoch + 1, "train_loss": running_loss, "train_accuracy": train_acc})
    print(f"Epoch {epoch+1} Summary - Loss: {running_loss:.4f}, Train Accuracy: {train_acc:.4f}")

    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
      for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        val_correct += (preds == labels).sum().item()
        val_total += labels.size(0)

    val_acc = val_correct / val_total
    wandb.log({"epoch": epoch + 1, "val_accuracy": val_acc})
    print(f"Validation Accuracy: {val_acc:.4f}")


In [ ]:
train_model(alexnet, criterion, optimizer, train_loader, val_loader, epochs=config.epochs)


Epoch 1/10
------------------------------
[Batch 10/251] Loss: 1.3068, Batch Acc: 0.4375
[Batch 20/251] Loss: 1.1510, Batch Acc: 0.6250
[Batch 30/251] Loss: 1.2605, Batch Acc: 0.6250
[Batch 40/251] Loss: 1.1965, Batch Acc: 0.6250
[Batch 50/251] Loss: 0.7559, Batch Acc: 0.8125
[Batch 60/251] Loss: 0.4293, Batch Acc: 0.8750
[Batch 70/251] Loss: 0.7865, Batch Acc: 0.6875
[Batch 80/251] Loss: 0.4854, Batch Acc: 0.8125
[Batch 90/251] Loss: 0.2071, Batch Acc: 0.9375
[Batch 100/251] Loss: 0.3397, Batch Acc: 0.8750
[Batch 110/251] Loss: 1.3512, Batch Acc: 0.8125
[Batch 120/251] Loss: 0.7856, Batch Acc: 0.6875
[Batch 130/251] Loss: 0.6702, Batch Acc: 0.6875
[Batch 140/251] Loss: 0.4029, Batch Acc: 0.7500
[Batch 150/251] Loss: 0.8946, Batch Acc: 0.6875
[Batch 160/251] Loss: 0.4652, Batch Acc: 0.8125
[Batch 170/251] Loss: 0.7364, Batch Acc: 0.8750
[Batch 180/251] Loss: 0.9895, Batch Acc: 0.8125
[Batch 190/251] Loss: 1.3584, Batch Acc: 0.6250
[Batch 200/251] Loss: 0.3161, Batch Acc: 0.8125
[Batch

KeyboardInterrupt: 